In [ ]:
import kagglehub
import os
import pandas as pd
from PIL import Image
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset
import numpy as np
from torch.utils.data import DataLoader, random_split
import glob

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO


# TO DO
class WaterDataset(Dataset):
  def __init__(self, image_dir, mask_dir, transform=None, target_transform=None):
        self.image_paths = glob.glob(os.path.join(image_dir, "*.jpg"))  # Get all image paths
        self.mask_paths = glob.glob(os.path.join(mask_dir, "*.png"))  # Get all image paths
        self.transform = transform
        self.target_transform = target_transform

        #sort() ensures images and masks are paired correctly by matching alphabetical order.
        self.image_paths.sort()
        self.mask_paths.sort()

  def __len__(self):
    # YOUR CODE HERE
    return len(self.image_paths)

  def __getitem__(self, idx):
    # TODO: Load the image and mask at index idx
      image = Image.open(self.image_paths[idx]).convert("RGB")
      mask = Image.open(self.mask_paths[idx]).convert("L")


    # Apply transforms
      if self.transform:
        image = self.transform(image)

      if self.target_transform:
        mask = self.target_transform(mask)
      mask = remap_mask(mask)

      return image, mask

image_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
        transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], # These random ahh values were found using ImageNet dataset, https://paperswithcode.com/dataset/imagenet
                          std=[0.2023, 0.1994, 0.2010])
])

# TODO: Add transforms
# Hint: Resize to (256, 256) with NEAREST interpolation, PILToTensor

# YOUR CODE HERE
mask_transforms = mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),  # NEAREST to preserve class indices
    transforms.PILToTensor()            # Convert mask to Tensor (keeps integer labels)
])


# Load training dataset
dataset = WaterDataset(os.path.join(path, 'dataset','images'), os.path.join(path, 'train','masks') , transform=image_transforms, target_transform=mask_transforms)
val_dataset = WaterDataset(os.path.join(path, 'dataset','images') , os.path.join(path, 'val','masks') , transform=image_transforms, target_transform=mask_transforms)
# split
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

print(f"Total: {len(dataset)}, Train: {len(train_dataset)}, Test: {len(test_dataset)}")
train_path = os.path.join(path, "Image")
train_path = os.path.join(path, "Mask")

# Create Train & Test DataLoaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2)

# Check dataset sizes
print(f"Training Samples: {len(train_dataset)}, Testing Samples: {len(test_dataset)}")

In [ ]:
def denormalize(img):
  mean = np.array([0.485, 0.456, 0.406])
  std = np.array([0.229, 0.224, 0.225])
  img = img.permute(1, 2, 0).numpy()  # CHW -> HWC
  img = img * std + mean
  img = np.clip(img, 0, 1)
  return img

In [ ]:
# Display some images with their masks
for i in range(3):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.permute(1, 2, 0), cmap="gray")
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")
    plt.show()


In [ ]:
!pip install -q segmentation_models_pytorch

In [ ]:
# TO DO
import segmentation_models_pytorch as smp

# Define U-Net Model
device = "cpu"
model = smp.Unet(
    encoder_name="efficientnet-b1",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8,  # Binary segmentation (1 output channel (e.g cat and background)) For MC classes=N (No. Objects, e.g cat and dog and background = 3)
).to(device)

In [ ]:
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm

# 🔹 Training Loop
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)  # mask shape becomes [N, H, W]
        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# 🔹 Validation Loop
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).squeeze(dim=1).to(torch.long)    # mask shape becomes [N, H, W]
            outputs = model(images)  # Now [N, H, W]
            loss = criterion(outputs, masks)
            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss() # OR LR = 0.001
optimizer = optim.AdamW(model.parameters(), lr=0.0001)

num_epochs = 10  # Define number of epochs
train_losses = []
val_losses = []

# Training Loop
for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")
# If doesnt move u may reduce BZ, No. Epochs, increase LR (0.0001 --> 0.001)

In [ ]:
### **🔹 Plot Training Loss Curve**
import matplotlib.pyplot as plt

plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np

# Function to denormalize images
def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
    std = np.array([0.229, 0.224, 0.225])  # ImageNet std
    img = img.numpy().transpose(1, 2, 0)  # Convert to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip values to [0,1]
    return img

# Set model to evaluation mode
model.eval()

# Get some test samples
test_samples = random.sample(range(len(val_dataset)), 5)

for idx in test_samples:
    img, mask = val_dataset[idx]

    with torch.no_grad():
            pred = model(img.unsqueeze(0).to(device))
            pred_mask = torch.argmax(torch.softmax(pred,dim=1), dim=1).cpu().squeeze()


    # Display images
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Original Image (Denormalized)
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    # Ground Truth Mask
    axes[1].imshow(mask.squeeze(), cmap="gray")
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    # Predicted Mask
    axes[2].imshow(pred_mask, cmap="gray")
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")

    plt.show()
